In [4]:
import pandas as pd

In [5]:
df_cus = pd.read_csv('customer_purchases.csv')
df_cus.sample(50)

,transaction_id,customer_id,customer_name,purchase_date,product_category,channel,purchase_value
1100,TXN001981,CUST0202,Dustin Beck,2021-09-26,Sports,Online,139.80
2237,TXN002310,CUST0237,Marc Wood,10-08-2020,Beauty,Online,98.98
2126,TXN001397,CUST0142,Kim James,2025-08-28,Fashion,In-Store,132.00
1434,TXN003414,CUST0379,Richard Khan,10/10/2020,Fashion,Online,21.07
2640,TXN001877,CUST0192,Rachel Ellis,2026-03-04,Beauty,Online,52.50
1739,TXN003739,CUST0415,Tasha Wood,2023-02-01,Fashion,In-Store,76.42
3939,TXN000269,CUST0029,David Walker,06/23/2023,Sports,Online,40.75
507,TXN002440,CUST0263,Bradley Hernandez,03-11-2021,Electronics,In-Store,220.78
3554,TXN003515,CUST0391,Taylor Smith,12/24/2023,Electronics,In-Store,60.58
2144,TXN001471,CUST0153,Katrina Alexander,2024-11-26,Home & Kitchen,Online,89.21


In [6]:
df_bran = pd.read_csv('branches.csv')
df_bran.sample(5)

,transaction_id,branch_id,branch_name,region
3915,TXN003643,BR01,Downtown,Central
810,TXN001400,BR04,Airport Plaza,West
1590,TXN003294,BR01,Downtown,Central
1826,TXN002825,BR01,Downtown,Central
3129,TXN000945,BR04,Airport Plaza,West


In [7]:
df_cus.dtypes

transaction_id          str
customer_id             str
customer_name           str
purchase_date           str
product_category        str
channel                 str
purchase_value      float64
dtype: object

In [8]:
import re
from datetime import datetime

def parse_dirty_date(date_str):
    if pd.isna(date_str) or str(date_str).strip() == '':
        return pd.NaT
    
    date_str = str(date_str).strip()
    
    formats = [
        '%Y-%m-%d',
        '%m/%d/%Y',
        '%m/%d/%y',
        '%d-%m-%Y',
        '%d/%m/%Y',
        '%d %b %Y',
        '%B %d, %Y',
    ]
    
    for fmt in formats:
        try:
            return pd.to_datetime(date_str, format=fmt)
        except:
            pass
    
    try:
        return pd.to_datetime(date_str, infer_datetime_format=True)
    except:
        pass
    
    if re.match(r'^\d+$', date_str):
        try:
            excel_epoch = pd.Timestamp('1899-12-30')
            return excel_epoch + pd.Timedelta(days=int(date_str))
        except:
            pass
    
    return pd.NaT

df_cus['purchase_date'] = df_cus['purchase_date'].apply(parse_dirty_date)

In [18]:
df_cus.dtypes

transaction_id                 str
customer_id                    str
customer_name                  str
purchase_date       datetime64[us]
product_category               str
channel                        str
purchase_value             float64
dtype: object

In [25]:
df_cus['purchase_value'] = pd.to_numeric(df_cus['purchase_value'])

In [10]:
df_cus.sample(10)

,transaction_id,customer_id,customer_name,purchase_date,product_category,channel,purchase_value
1405,TXN002164,CUST0218,Breanna Cooper,NaT,Home & Kitchen,In-Store,136.09
3698,TXN000156,CUST0012,Robert Collins,NaT,Fashion,Online,37.09
2607,TXN001892,CUST0193,Samantha Alexander,NaT,Electronics,Online,52.63
3661,TXN003037,CUST0330,Richard Smith,NaT,Home & Kitchen,Online,60.67
2585,TXN002602,CUST0279,Ian Haney,2021-06-08,Electronics,In-Store,45.23
3831,TXN002026,CUST0204,Beverly Stanley,2023-05-19,Electronics,Online,43.91
1059,TXN002119,CUST0213,Dillon Fletcher,2021-03-17,Sports,In-Store,100.76
2378,TXN000260,CUST0028,Kristin Strong,2020-06-26,Home & Kitchen,Online,75.32
580,TXN003606,CUST0400,Cindy Reeves,2023-07-05,Fashion,Online,26.68
2336,TXN002324,CUST0243,Kenneth Jackson III,2021-08-08,Electronics,Online,123.70


In [21]:
from sqlalchemy import create_engine
DB_CONFIG = {
    'host': 'localhost',
    'port': '5432',
    'database': 'Nova_Retail',
    'user': 'postgres',
    'password': 'Alahly1907'
}

def get_engine():
  
    url = (
        f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )
    return create_engine(url)

In [26]:
engine = get_engine()

df_cus.to_sql('customer_purchase', engine, if_exists= 'replace',index=False)

21

In [ ]:
df_bran.to_sql('branches_purchase', engine, if_exists='replace', index=False)

21

In [ ]:
#df_all = pd.read_sql_table("fact_employee", con=engine)
#